# ViroWatch Neo4j bulk loader

Reads the `kg/` CSVs produced by ViroWatch step 11 and bulk-merges them into a
Neo4j knowledge graph.  All operations are idempotent — re-running this
notebook on the same results directory is safe.

**Load order** (respects FK-like dependencies):
1. `sample.csv` → `Sample`
2. `assembly.csv` → `Assembly` + `(Sample)-[:HAS_ASSEMBLY]->(Assembly)`
3. `biodata_files.csv` → `BioDataFile` + `PRODUCE` / `ASSEMBLED_FROM` edges
4. `contigs.csv` → `Contig` + `(BioDataFile)-[:HAS_CONTIG]->(Contig)`
5. `stanford_alignments.csv` → `StanfordHIVDRAlignment` + `Protein` + contig edge
6. `stanford_predictions.csv` → `StanfordHIVDRPrediction` + `Drug` + `DrugClass` + contig/sample edges
7. `mutations.csv` → `Mutation` + `(StanfordHIVDRAlignment)-[:FOUND]->(Mutation)`
8. `blast_hits.csv` → `ReferenceGenome` + `Organism` + `(Contig)-[:HAS_BLAST_HIT]->(ReferenceGenome)`

In [ ]:
%pip install neo4j --quiet

In [ ]:
import csv
import time
from pathlib import Path
from neo4j import GraphDatabase

## Configuration

Edit the three variables below before running.

In [ ]:
NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"   # change to your Neo4j password

# Root results directory — the notebook scans every sub-directory that
# contains a kg/ folder (one sub-directory = one ViroWatch sample).
RESULTS_DIR = Path("../results")   # adjust to your --outdir value

# How many records to send per transaction.  500 is a good default;
# lower this if Neo4j runs out of heap on very large contig sequences.
BATCH_SIZE = 500

## Connection test

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
driver.verify_connectivity()
print("Connected to", NEO4J_URI)

## Helpers

In [ ]:
def read_csv(path):
    """Read a CSV and return a list of dicts; empty strings become None."""
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as fh:
        rows = []
        for row in csv.DictReader(fh):
            rows.append({k: (None if v == "" else v) for k, v in row.items()})
    return rows


def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i : i + size]


def run_batched(session, cypher, records, batch_size=BATCH_SIZE):
    """Execute *cypher* in batches, passing each batch as $records.

    Returns total (nodes_created, relationships_created).
    """
    nodes_created = rels_created = 0
    for batch in chunked(records, batch_size):
        result = session.run(cypher, records=batch)
        summary = result.consume()
        nodes_created += summary.counters.nodes_created
        rels_created  += summary.counters.relationships_created
    return nodes_created, rels_created


def discover_samples(results_dir):
    """Return a sorted list of (sample_id, kg_dir) for every kg/ directory found."""
    found = []
    for d in sorted(Path(results_dir).iterdir()):
        kg = d / "kg"
        if d.is_dir() and kg.is_dir():
            found.append((d.name, kg))
    return found


samples = discover_samples(RESULTS_DIR)
print(f"Found {len(samples)} sample(s):")
for sid, kgd in samples:
    csvs = [p.name for p in sorted(kgd.glob("*.csv"))]
    print(f"  {sid}: {', '.join(csvs) or '(no CSVs)'}")

## Cypher queries

One query per CSV type.  All use `UNWIND $records AS rec` + `MERGE` so they are
idempotent.

In [ ]:
# ── 1. Sample ─────────────────────────────────────────────────────────────────
CQ_SAMPLE = """
UNWIND $records AS rec
MERGE (s:Sample {sample_id: rec.sample_id})
  ON CREATE SET s.created_at = datetime()
"""

# ── 2. Assembly ───────────────────────────────────────────────────────────────
CQ_ASSEMBLY = """
UNWIND $records AS rec
MERGE (a:Assembly {assembly_id: rec.assembly_id})
  ON CREATE SET a.assembler = rec.assembler,
                a.created_at = date(rec.created_at)
WITH a, rec
MATCH (s:Sample {sample_id: rec.sample_id})
MERGE (s)-[:HAS_ASSEMBLY]->(a)
"""

# ── 3a. BioDataFile — FASTA (produced by assembly) ────────────────────────────
CQ_BIODATA_FASTA = """
UNWIND $records AS rec
MERGE (f:BioDataFile {uri: rec.uri})
  ON CREATE SET f.file_type  = rec.file_type,
                f.compressed = rec.compressed,
                f.sha256     = rec.sha256
WITH f, rec
MATCH (a:Assembly {assembly_id: rec.assembly_id})
MERGE (a)-[:PRODUCE]->(f)
"""

# ── 3b. BioDataFile — FASTQ (input reads consumed by assembly) ────────────────
CQ_BIODATA_FASTQ = """
UNWIND $records AS rec
MERGE (f:BioDataFile {uri: rec.uri})
  ON CREATE SET f.file_type  = rec.file_type,
                f.compressed = rec.compressed,
                f.sha256     = rec.sha256
WITH f, rec
MATCH (s:Sample {sample_id: rec.sample_id})-[:HAS_ASSEMBLY]->(a:Assembly)
MERGE (a)-[:ASSEMBLED_FROM]->(f)
"""

# ── 4. Contig ──────────────────────────────────────────────────────────────────
# After merge, link to the FASTA BioDataFile via the assembly.
CQ_CONTIG = """
UNWIND $records AS rec
MERGE (c:Contig {contig_id: rec.contig_id})
  ON CREATE SET
    c.contig_name        = rec.contig_name,
    c.length             = toInteger(rec.length),
    c.coverage           = toFloat(rec.coverage),
    c.is_circular        = (rec.is_circular = 'true'),
    c.is_repeated_region = (rec.is_repeated_region = 'true'),
    c.multiplicity       = toInteger(rec.multiplicity),
    c.alt_group          = rec.alt_group,
    c.graph_path         = rec.graph_path,
    c.sequence           = rec.sequence,
    c.sequence_hash      = rec.sequence_hash,
    c.hash_algorithm     = rec.hash_algorithm
WITH c, rec
MATCH (a:Assembly {assembly_id: rec.assembly_id})-[:PRODUCE]->(f:BioDataFile)
MERGE (f)-[:HAS_CONTIG]->(c)
"""

# ── 5. Stanford alignments ────────────────────────────────────────────────────
# Creates StanfordHIVDRAlignment + Protein nodes and the ALIGNED_TO edge.
# Then links the Contig (via contig_id_prefix) to the alignment.
CQ_STANFORD_ALIGNMENTS = """
UNWIND $records AS rec
MERGE (al:StanfordHIVDRAlignment {result_sha256: rec.result_sha256})
  ON CREATE SET
    al.for_contig_id = rec.contig_id_prefix,
    al.timestamp = CASE
      WHEN rec.timestamp IS NOT NULL AND trim(rec.timestamp) <> ''
      THEN datetime(rec.timestamp) ELSE NULL END,
    al.database_version       = rec.database_version,
    al.database_published_date = rec.database_published_date
WITH al, rec
FOREACH (_ IN CASE WHEN rec.gene IS NOT NULL THEN [1] ELSE [] END |
  MERGE (p:Protein {abbreviation: rec.gene})
  MERGE (al)-[:ALIGNED_TO]->(p)
)
WITH al, rec
OPTIONAL MATCH (c:Contig {contig_id: rec.contig_id_prefix})
FOREACH (_ IN CASE WHEN c IS NOT NULL THEN [1] ELSE [] END |
  MERGE (c)-[:HAS_STANFORD_HIVDR_ALIGNMENT]->(al)
)
"""

# ── 6. Stanford predictions (ARV resistance panel) ────────────────────────────
# Matches BULK_MERGE_ARVPredictions.cypher + adds Contig edge.
CQ_STANFORD_PREDICTIONS = """
UNWIND $records AS rec
MERGE (dc:DrugClass {name: rec.drug_class})
MERGE (d:Drug {name: rec.drug_name})
  ON CREATE SET d.full_name    = rec.drug_full_name,
                d.display_abbr = rec.drug_abbr
MERGE (d)-[:IN_DRUG_CLASS]->(dc)
MERGE (pred:StanfordHIVDRPrediction {prediction_id: rec.prediction_id})
SET pred.sample_id       = rec.sample_id,
    pred.gene            = rec.gene,
    pred.drug_name       = rec.drug_name,
    pred.score           = rec.score,
    pred.level           = rec.level,
    pred.interpretation  = rec.interpretation
MERGE (pred)-[:PREDICTS_RESISTANCE_TO]->(d)
WITH pred, rec
MATCH (s:Sample {sample_id: rec.sample_id})
MERGE (s)-[:HAS_STANFORD_HIVDR_PREDICTION]->(pred)
WITH pred, rec
OPTIONAL MATCH (c:Contig {contig_id: rec.contig_id})
FOREACH (_ IN CASE WHEN c IS NOT NULL THEN [1] ELSE [] END |
  MERGE (c)-[:HAS_STANFORD_HIVDR_PREDICTION]->(pred)
)
"""

# ── 7. Mutations ──────────────────────────────────────────────────────────────
# Matches BULK_MERGE_StanfordMutations.cypher.
CQ_MUTATIONS = """
UNWIND $records AS rec
MERGE (m:Mutation {gene: rec.gene, text: rec.text})
SET m.mutation_id       = rec.mutation_id,
    m.primary_type      = rec.primary_type,
    m.is_sdrm           = rec.is_sdrm,
    m.position          = rec.position,
    m.has_stop          = rec.has_stop,
    m.is_apobec_mutation = rec.is_apobec_mutation,
    m.is_apobec_drm     = rec.is_apobec_drm,
    m.is_insertion      = rec.is_insertion,
    m.is_deletion       = rec.is_deletion,
    m.is_unusual        = rec.is_unusual
FOREACH (_ IN CASE WHEN rec.result_sha256 IS NOT NULL THEN [1] ELSE [] END |
  MERGE (al:StanfordHIVDRAlignment {result_sha256: rec.result_sha256})
  MERGE (al)-[:FOUND]->(m)
)
"""

# ── 8. BLAST hits ─────────────────────────────────────────────────────────────
# Matches BULK_MERGE_BLAST_HITS.cypher + adds Contig edge.
CQ_BLAST_HITS = """
UNWIND $records AS rec
MERGE (refg:ReferenceGenome {accession_no: rec.accession_no})
  ON CREATE SET refg.name            = rec.name,
                refg.source_database = rec.source_database,
                refg.annotation_source = 'BLAST'
FOREACH (_ IN CASE WHEN rec.taxid IS NOT NULL THEN [1] ELSE [] END |
  MERGE (o:Organism {taxid: rec.taxid})
    ON CREATE SET o.sciname = rec.sciname
  MERGE (refg)-[:REFERENCE_GENOME_OF]->(o)
)
WITH refg, rec
OPTIONAL MATCH (c:Contig {contig_id: rec.query_contig_id})
FOREACH (_ IN CASE WHEN c IS NOT NULL THEN [1] ELSE [] END |
  MERGE (c)-[:HAS_BLAST_HIT]->(refg)
)
"""

print("Cypher queries defined.")

## Load all samples

In [ ]:
total_nodes = total_rels = 0
t0 = time.time()

for sample_id, kg_dir in samples:
    print(f"\n── {sample_id} ──")
    sample_nodes = sample_rels = 0

    with driver.session() as sess:

        # 1. Sample
        rows = read_csv(kg_dir / "sample.csv")
        if rows:
            n, r = run_batched(sess, CQ_SAMPLE, rows)
            print(f"  sample.csv            +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 2. Assembly
        rows = read_csv(kg_dir / "assembly.csv")
        if rows:
            n, r = run_batched(sess, CQ_ASSEMBLY, rows)
            print(f"  assembly.csv          +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 3. BioDataFile — split by file_type
        rows = read_csv(kg_dir / "biodata_files.csv")
        if rows:
            fasta_rows = [r for r in rows if r.get("file_type") == "FASTA" and r.get("assembly_id")]
            fastq_rows = [r for r in rows if r.get("file_type") == "FASTQ"]
            if fasta_rows:
                n, r = run_batched(sess, CQ_BIODATA_FASTA, fasta_rows)
                print(f"  biodata_files FASTA   +{n} nodes  +{r} rels")
                sample_nodes += n; sample_rels += r
            if fastq_rows:
                n, r = run_batched(sess, CQ_BIODATA_FASTQ, fastq_rows)
                print(f"  biodata_files FASTQ   +{n} nodes  +{r} rels")
                sample_nodes += n; sample_rels += r

        # 4. Contigs
        rows = read_csv(kg_dir / "contigs.csv")
        if rows:
            n, r = run_batched(sess, CQ_CONTIG, rows)
            print(f"  contigs.csv           +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 5. Stanford alignments
        rows = read_csv(kg_dir / "stanford_alignments.csv")
        if rows:
            n, r = run_batched(sess, CQ_STANFORD_ALIGNMENTS, rows)
            print(f"  stanford_alignments   +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 6. Stanford predictions
        rows = read_csv(kg_dir / "stanford_predictions.csv")
        if rows:
            n, r = run_batched(sess, CQ_STANFORD_PREDICTIONS, rows)
            print(f"  stanford_predictions  +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 7. Mutations
        rows = read_csv(kg_dir / "mutations.csv")
        if rows:
            n, r = run_batched(sess, CQ_MUTATIONS, rows)
            print(f"  mutations.csv         +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

        # 8. BLAST hits
        rows = read_csv(kg_dir / "blast_hits.csv")
        if rows:
            n, r = run_batched(sess, CQ_BLAST_HITS, rows)
            print(f"  blast_hits.csv        +{n} nodes  +{r} rels")
            sample_nodes += n; sample_rels += r

    print(f"  → {sample_nodes} nodes  {sample_rels} relationships")
    total_nodes += sample_nodes
    total_rels  += sample_rels

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — {total_nodes} nodes created, {total_rels} relationships created")

## Spot-check queries

Quick sanity checks — run after loading to verify the graph is wired correctly.

In [ ]:
checks = [
    ("Samples",                "MATCH (s:Sample) RETURN count(s) AS n"),
    ("Assemblies",             "MATCH (a:Assembly) RETURN count(a) AS n"),
    ("Contigs",                "MATCH (c:Contig) RETURN count(c) AS n"),
    ("BioDataFiles",           "MATCH (f:BioDataFile) RETURN count(f) AS n"),
    ("DR Predictions",         "MATCH (p:StanfordHIVDRPrediction) RETURN count(p) AS n"),
    ("Mutations",              "MATCH (m:Mutation) RETURN count(m) AS n"),
    ("Reference Genomes",      "MATCH (r:ReferenceGenome) RETURN count(r) AS n"),
    ("Sample→Assembly edges",  "MATCH ()-[:HAS_ASSEMBLY]->() RETURN count(*) AS n"),
    ("Contig→Prediction edges","MATCH ()-[:HAS_STANFORD_HIVDR_PREDICTION]->() RETURN count(*) AS n"),
    ("Contig→BLAST edges",     "MATCH ()-[:HAS_BLAST_HIT]->() RETURN count(*) AS n"),
]

with driver.session() as sess:
    for label, cq in checks:
        result = sess.run(cq)
        print(f"  {label:<28} {result.single()['n']:>6}")

In [ ]:
# Per-sample resistance summary
summary_cq = """
MATCH (s:Sample)-[:HAS_STANFORD_HIVDR_PREDICTION]->(pred:StanfordHIVDRPrediction)
       -[:PREDICTS_RESISTANCE_TO]->(d:Drug)-[:IN_DRUG_CLASS]->(dc:DrugClass)
WHERE toInteger(pred.level) >= 3
RETURN s.sample_id AS sample,
       dc.name     AS drug_class,
       d.name      AS drug,
       pred.interpretation AS resistance
ORDER BY sample, drug_class, drug
"""
with driver.session() as sess:
    records = sess.run(summary_cq).data()

if records:
    print(f"{'Sample':<20} {'DrugClass':<10} {'Drug':<8} Resistance")
    print("-" * 60)
    for rec in records:
        print(f"{rec['sample']:<20} {rec['drug_class']:<10} {rec['drug']:<8} {rec['resistance']}")
else:
    print("No resistance levels ≥ 3 found.")

In [ ]:
driver.close()
print("Connection closed.")